# 05 · Can CLIP sort images into classes it was never trained on?

Can you classify images without training a classifier? With CLIP, you can: write one sentence per label, like *"A photo of a cat."*, and pick whichever sentence sits closest to the image. That's **zero-shot classification**, and we'll try it on 100 CIFAR-10 test images.

**Read more in [the book](https://arxiv.org/abs/2605.07544):** Chapter 1, Section 1.4.3, *Fusion and Alignment Mechanisms*, and Chapter 5, Section 5.4.2, *Robustness and Out-of-Distribution Evaluation*.

> **How to use this notebook:** Run the cells top to bottom. When you hit a **Pause and predict**, take a guess before running the next cell. You'll remember way more that way. Cells marked **TODO** are yours to play with: change a value, re-run, and see what happens.

### Pick your GPU

Got a GPU (or a few)? Set `GPU_ID` in the next cell to the one you want. Run `nvidia-smi` in a terminal to see the list. Only have one? Leave it at `"0"`. No GPU at all? Set it to `""` and everything runs on the CPU. It's slower, but it works.

This cell has to run *before* PyTorch is imported, so if you change it later, restart the kernel and run from the top.

In [ ]:
# TODO: choose your GPU. This has to run before torch is imported.
#   "0"  -> the first GPU (leave this if you only have one)
#   "1"  -> the second GPU, and so on (`nvidia-smi` lists them)
#   ""   -> no GPU, run everything on the CPU
#   None -> don't restrict anything; PyTorch sees every GPU
GPU_ID = "0"

import os
if GPU_ID is not None:
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # number GPUs the same way nvidia-smi does
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 1. What do we need to get started?

### 1.1 Grab our tools

The usual crew: Matplotlib for pictures, PyTorch for tensors, `datasets` for CIFAR-10, and a pretrained CLIP from `transformers`. The dataset gives us images *and* labels, but here's the deal: the labels never go into the model. We only use them at the end to grade CLIP's answers.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoProcessor, CLIPModel  # one checkpoint gives us both encoders

### 1.2 Pin down the data, the model, and the sample

CIFAR-10 is the same tiny-photo dataset from Notebook 01: ten everyday categories, 32 × 32 pixels each. This time we use the **test** split.

A few settings worth knowing about:

- `DATASET_REVISION` pins an exact version of the dataset, so the records never shift under us.
- `SAMPLE_SIZE = 100`: small enough to eyeball, big enough to include real wins *and* real mistakes.
- `SEED = 23`: the random seed, so we pick the same 100 images every time we run this.

In [ ]:
DATASET_ID, DATASET_CONFIG = "uoft-cs/cifar10", "plain_text"
DATASET_REVISION = "0b2714987fa478483af9968de7c934580d0bb9a2"  # pin the exact dataset version

MODEL_ID = "openai/clip-vit-base-patch32"
SAMPLE_SIZE, SEED = 100, 23

One more bit of setup: where should the model and its inputs live? GPU if we have one, CPU otherwise.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU if we have one, CPU otherwise
print("device:", device)  # "cuda" here means the GPU we picked at the top

## 2. What's the classification problem?

### 2.1 Load the test images and their class names

Where do the ten class names come from? We could type them out by hand, but it's safer to read them straight from the dataset. That way their order is guaranteed to match the integer labels: class `0` is the first name in the list, class `1` the second, and so on.

In [ ]:
test_data = load_dataset(
    DATASET_ID, DATASET_CONFIG, split="test", revision=DATASET_REVISION
)
class_names = test_data.features["label"].names  # position in this list = class ID
print(test_data)
print("classes:", class_names)

### 2.2 Pick 100 images (before we look at any scores)

The seeded random generator picks the same 100 positions every run. We keep the images and labels in matching order, so `images[i]` goes with `labels[i]`. And again: the labels sit off to the side. They never touch the model.

Why pick the sample *before* looking at any scores? So we can't accidentally cherry-pick images that make CLIP look good.

In [ ]:
sample_generator = torch.Generator().manual_seed(SEED)  # its own generator, so the model can't affect which images we get
indices = torch.randperm(len(test_data), generator=sample_generator)[:SAMPLE_SIZE]
records = [test_data[int(index)] for index in indices]
images = [record["img"].convert("RGB") for record in records]
labels = torch.tensor([record["label"] for record in records])
print("selected test images:", len(images))

Let's look at a few before CLIP does. Remember, these are only 32 × 32 pixels, so set your expectations accordingly. A truck and an automobile can look a lot alike at this size, and a small animal might be just a few blurry blobs.

The titles here are the dataset's labels, not predictions. CLIP hasn't seen anything yet.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4.5))  # titles are the true labels, not predictions
for axis, image, label in zip(axes.flat, images[:8], labels[:8]):
    axis.imshow(image)
    axis.set_title(class_names[int(label)])
    axis.axis("off")
fig.suptitle("Real CIFAR-10 test records")
fig.tight_layout()
plt.show()

## 3. How do we turn the images into vectors?

This part should feel familiar from Notebook 01. We're just doing it for 100 images at once.

### 3.1 Load CLIP

Same pair as always, both from the same checkpoint name:

- the **processor** knows how to resize and normalize images (and tokenize text) the way this checkpoint expects
- the **model** holds two encoders, one for images and one for text, that were trained together so their vectors can be compared

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)  # the prep recipe that belongs to this checkpoint
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()  # we're only using the model, not training it

### 3.2 Prep all 100 images as one batch

The processor blows each tiny image up to the resolution CLIP expects and stacks them all into one tensor shaped `[image, channel, height, width]`. The first number should be 100.

In [ ]:
image_inputs = processor(images=images, return_tensors="pt").to(device)
print("pixel batch:", tuple(image_inputs["pixel_values"].shape))  # first dimension should be 100

### 3.3 Get one unit-length vector per image

`get_image_features` runs the whole vision side and the projection into the shared space in one call, so we skip the step-by-step tour from Notebook 01. Then we L2-normalize each row to length 1, which means a plain dot product later is the cosine similarity.

In [ ]:
with torch.inference_mode():  # no gradients needed, we're just reading features
    image_output = model.get_image_features(**image_inputs)
image_features = F.normalize(image_output.pooler_output, dim=-1)  # length 1 per row
print("image vectors:", tuple(image_features.shape))

## 4. How do class names become a classifier?

Here's the heart of the whole notebook. It's exactly what panels (2) and (3) of Figure 1 in the [CLIP paper](https://arxiv.org/abs/2103.00020) show:

<img src="../assets/diagrams/notebook3/fig1.png" alt="CLIP paper Figure 1: contrastive pre-training, building a classifier from label text, and zero-shot prediction" width="800">

*Figure 1 from Radford et al. (2021).*

- **(1) Contrastive pre-training** is how CLIP was trained. We rebuilt that in Notebook 03, and here we just use the result.
- **(2) Create dataset classifier from label text:** every class name goes into the same sentence, *"A photo of a {object}."*, and through the text encoder. We get one vector per class. That's this section.
- **(3) Use for zero-shot prediction:** the image goes through the image encoder, gets compared with every class vector, and the highest score wins. That's Section 5.

### 4.1 Write one sentence per class

CLIP was trained on images paired with real sentences, not on class IDs like `3`, and not really on lone words either. So instead of feeding it `"cat"`, we wrap each class name in a little sentence called a **prompt**. The sentence pattern is the **template**; here it's `"A photo of a {name}."`

Every class gets the exact same template, so the class name is the only thing that changes from prompt to prompt. The helper also picks "a" or "an" so the sentences read naturally.

In [ ]:
def class_prompt(name):
    article = "an" if name[0].lower() in "aeiou" else "a"  # "an airplane", "a bird"
    return f"A photo of {article} {name}."  # the prompt template (we'll play with it in a TODO below)

prompts = [class_prompt(name) for name in class_names]
for prompt in prompts:
    print(prompt)

Ten classes, ten sentences. Those ten sentences *are* our classifier. That's the whole idea of Figure 1 in the CLIP paper: build the classifier out of the label text.

### 4.2 Encode all ten sentences

First, tokenize. The sentences aren't all the same length, so `padding=True` fills the short ones up and we get a nice rectangular batch.

In [ ]:
text_inputs = processor(text=prompts, padding=True, return_tensors="pt").to(device)
print("token batch:", tuple(text_inputs["input_ids"].shape))  # 10 rows, one per class

These are still just token IDs, positions in a vocabulary, so they're not class vectors yet. The text encoder has to read each prompt in context, pool it into one summary, and project it into the shared space (Notebook 02 walked through each of those steps). Then we normalize, just like we did for the images.

In [ ]:
with torch.inference_mode():
    text_output = model.get_text_features(**text_inputs)
class_features = F.normalize(text_output.pooler_output, dim=-1)  # length 1, so dot products are cosine scores
print("class vectors:", tuple(class_features.shape))
print("shared width:", image_features.shape[-1])

Check that the last numbers match. We've got 100 image vectors and 10 class vectors, all the same width and all in the same shared space. That's everything we need to compare them.

## 5. How does CLIP make the call?

### 5.1 Compare every image with every class

One matrix multiply does it all. `[100, width]` times `[width, 10]` gives a `[100, 10]` **score table**. Row `i` holds image `i`'s cosine similarity with each of the ten class sentences.

In [ ]:
cosine_scores = image_features @ class_features.T  # [100, width] @ [width, 10] -> [100, 10], columns in class order
print("score matrix [images, classes]:", tuple(cosine_scores.shape))

### 5.2 Turn each row into probabilities

Raw cosine scores tend to sit close together, which makes them hard to read. So we do two things to each row:

1. Multiply by CLIP's **logit scale**, a positive number it learned during pretraining. It's the same "learned temperature" idea we trained ourselves in Notebook 04. A bigger scale stretches the scores apart.
2. Apply a **softmax**, which turns the ten scores into ten positive numbers that add up to 1.

> **Pause and predict:** Can multiplying by the scale ever change which class wins for an image? Think about what multiplying by a positive number does to the order of a list.

In [ ]:
with torch.inference_mode():
    logit_scale = model.logit_scale.exp()  # stored as a log, so exp() gives the actual scale
    logits = logit_scale * cosine_scores  # a positive scale never changes which score is biggest
    probabilities = logits.softmax(dim=-1)
print("learned scale:", round(logit_scale.item(), 2))

Nope, it can't. Multiplying by a positive number keeps every score in the same order, and softmax keeps the order too. So the winner stays the winner. The scale only changes how *sharp* the probabilities look: a big scale makes the top class look more decisive, a small one flattens everything out. (In the CLIP paper, this scale was capped at 100 during training, so it'll probably land right around there.)

> **Heads up:** These probabilities only compare the ten sentences *we* wrote. If you swap a prompt or add an eleventh class, every number changes. So read them as "how CLIP splits its vote among these options", not as a calibrated confidence like "90% sure this is a cat".

### 5.3 Pick the winner for each image

`max` gives us two things per row: the biggest probability, and the column it sits in. Since we never shuffled the prompts, that column number is also the class ID. We compare it with the true labels to get accuracy.

> **Pause and predict:** With ten classes, random guessing gets about 10% right. CLIP has never been trained on a single labeled CIFAR-10 image. Where do you think it lands?

In [ ]:
confidence, predictions = probabilities.max(dim=-1)  # one winning column per image
correct = predictions.cpu().eq(labels)
accuracy = correct.float().mean()
print("zero-shot accuracy:", f"{accuracy:.1%}")
print("correct images:", int(correct.sum()), "of", len(correct))

Compare that with the 10% you'd get from guessing. Every correct answer came from nothing but ten sentences and a pretrained model. No CIFAR-10 training, no classifier head.

One thing to keep in mind: this is only 100 images, so each one is worth a full percentage point. Run a different sample and the number will wobble a bit.

Does the sentence around the class name really matter? The CLIP authors checked. Across 36 datasets, careful prompts plus averaging over several templates (called **prompt ensembling**) beat bare class names by almost 5 points, about the same gain as using a model with 4 times the compute:

<img src="../assets/diagrams/notebook5/clip_fig4.png" alt="CLIP paper Figure 4: prompt engineering and ensembling improve zero-shot performance" width="420">

*Figure 4 from Radford et al. (2021).*

You'll get to try both ideas yourself: the next cell for templates, and TODO 1 at the end for ensembling.

**TODO:** Here's where it gets fun. Is `"A photo of a {name}."` actually a good template? The cell below rebuilds the classifier from a template of your choice and re-scores the same 100 images. It starts with the bare class name, no sentence at all. Change `my_template` and re-run. It only re-encodes ten short sentences, so it's quick.

In [ ]:
# TODO: change the template and re-run. Keep the {} where the class name goes.
my_template = "{}"
my_prompts = [my_template.format(name) for name in class_names]
my_text_inputs = processor(text=my_prompts, padding=True, return_tensors="pt").to(device)
with torch.inference_mode():
    my_class_features = F.normalize(model.get_text_features(**my_text_inputs).pooler_output, dim=-1)
my_predictions = (image_features @ my_class_features.T).argmax(dim=-1)  # the scale can't change the winner, so skip it
my_accuracy = my_predictions.cpu().eq(labels).float().mean()
print("example prompt:", my_prompts[0])
print(f"accuracy: {my_accuracy:.1%}  (original template: {accuracy:.1%})")

Did the wording move the accuracy? On such a small sample, a difference of one or two images might just be noise, but bigger jumps are telling you something. Changing the words changes the class vectors, so it changes the classifier itself.

<details>
<summary>Hint: templates worth trying</summary>

- `"a photo of a {}."`: the CLIP paper's go-to template
- `"a blurry photo of a {}."` or `"a low resolution photo of a {}."`: these images *are* tiny and blurry, after all
- `"a drawing of a {}."`: a description that doesn't fit these images
- `"a photo of a {}, a type of animal."`: sounds fine for cats, but what does it do to trucks?

The CLIP paper found that a good template usually beats the bare class name. So prompt wording really is part of the experiment, not just decoration.

</details>

### 5.4 Look at the whole decision for one image

A single predicted label hides a lot. Was the winner way ahead? Were two related classes neck and neck? Or was everything basically flat? A bar chart of one image's ten probabilities shows us.

> **Pause and predict:** Scroll back up to the first image in the gallery. Which class do you think gets the tallest bar? Which one or two might come in second?

In [ ]:
fig, axis = plt.subplots(figsize=(7, 3.5))  # all ten candidates for the first image
axis.bar(class_names, probabilities[0].detach().cpu(), color="tab:blue")
axis.set(ylabel="probability within this prompt set", title="First image: all candidate descriptions")
axis.tick_params(axis="x", rotation=35)
axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

Check the tallest bar against the first image's true label from the gallery. Then look at the runners-up. If any classes got a decent share, are they ones that make sense for this picture? Close calls between look-alike classes are often more informative than the winner itself.

## 6. What do the hits and misses look like?

Accuracy is one number. Let's put actual decisions next to actual pictures. Each title shows the true label, CLIP's guess, and the probability it gave that guess. Green means correct, crimson means wrong.

When you spot a mistake, try to figure out *why*. Is the image just too small and blurry? Are the two classes genuinely similar? Or does the prompt not describe that category very well?

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(9, 5))  # color only marks right or wrong, it's not part of the scoring
for index, axis in enumerate(axes.flat):
    truth = class_names[int(labels[index])]
    guess = class_names[int(predictions[index])]
    color = "green" if correct[index] else "crimson"
    axis.imshow(images[index])
    title = f"true: {truth}" + chr(10) + f"pred: {guess} ({confidence[index]:.2f})"
    axis.set_title(title, color=color, fontsize=9)
    axis.axis("off")
fig.tight_layout()
plt.show()

Look at the numbers in parentheses too. Is CLIP ever wrong with a high probability? Is it ever right with a low one? That's the earlier heads-up in action: these probabilities tell you how CLIP split its vote among our ten sentences, not how often it's actually right.

> **Something to chew on:** CLIP learned from images paired with captions scraped from the internet. What sort of captions do you think come with photos like these? And would a person ever caption a 32 × 32 thumbnail "A photo of a deer."?

## Recap

That's the whole trip. We built a ten-class classifier out of nothing but sentences:

```text
10 class names                  "airplane", "automobile", ...
      ↓ prompt template
10 prompts                      "A photo of an airplane.", ...
      ↓ CLIP text encoder + normalize
class vectors                   [10, 512]

100 test images
      ↓ CLIP image encoder + normalize
image vectors                   [100, 512]

image vectors @ class vectors.T
      ↓
score table                     [100, 10]
      ↓ × logit scale -> softmax -> max
one predicted class per image   (labels only used for grading)
```

**The big ideas:**

- **Zero-shot classification is really just retrieval.** Each image picks the closest sentence out of a short list that *you* wrote.
- There's no trained classifier head. The text encoder builds the class vectors on the spot, from the prompts.
- **Prompt wording is part of the experiment.** Change the words and you change the class vectors, which means you change the classifier.
- The logit scale and softmax make the scores easier to read, but they never change the winner.
- Accuracy sums up the whole sample. Probability bars and error images explain the individual decisions.

### Check your understanding

1. We never trained anything here. So where exactly is the "classifier"? What plays the role of its weights?
2. Why does multiplying by `logit_scale` leave the accuracy exactly the same?
3. Say you want an eleventh class, `"pickup truck"`. What do you have to recompute, and what can you reuse as-is? What shape is the new score table?

<details>
<summary>Show answers</summary>

1. The ten normalized class vectors in `class_features` are the classifier. Scoring an image is a dot product with each of them, which is exactly what a linear classifier layer does. The weights just come from the text encoder instead of from training.
2. It's a positive number, so it scales every score in a row by the same factor and keeps their order. Softmax keeps the order too, so the class with the biggest score is the same either way.
3. You only need to encode one new prompt and add its vector as an extra row of `class_features`. The 100 image vectors don't change at all. The score table becomes `[100, 11]`.

</details>

## Try it yourself

### TODO 1: Average several templates (prompt ensembling)

The CLIP paper didn't stop at one template. For each class, it encoded the class name in lots of different templates and **averaged** the resulting vectors. This is called **prompt ensembling**. The idea: one wording might be a bit off, but the average of many wordings is more robust.

The cell below does that with four templates. It runs as-is. Add your own templates, remove some, and see what happens to the accuracy.

In [ ]:
# TODO: add, remove, or edit templates and re-run (keep the {} in each one)
my_templates = [
    "a photo of a {}.",
    "a blurry photo of a {}.",
    "a low resolution photo of a {}.",
    "a close-up photo of a {}.",
]
ensemble_rows = []
for template in my_templates:
    template_inputs = processor(
        text=[template.format(name) for name in class_names], padding=True, return_tensors="pt"
    ).to(device)
    with torch.inference_mode():
        template_features = F.normalize(model.get_text_features(**template_inputs).pooler_output, dim=-1)
    ensemble_rows.append(template_features)  # one [10, width] block per template

# average over templates, then re-normalize: [templates, 10, width] -> [10, width]
ensemble_features = F.normalize(torch.stack(ensemble_rows).mean(dim=0), dim=-1)
ensemble_predictions = (image_features @ ensemble_features.T).argmax(dim=-1)
ensemble_accuracy = ensemble_predictions.cpu().eq(labels).float().mean()
print(f"ensemble of {len(my_templates)} templates: {ensemble_accuracy:.1%}  (single template: {accuracy:.1%})")

<details>
<summary>Hint</summary>

Why re-normalize after averaging? The average of several length-1 vectors is usually a bit shorter than 1, and we want the dot product to stay a cosine similarity.

You might notice these templates say "a" even before "airplane" and "automobile". The CLIP paper's templates did the same, so we're keeping it simple here.

</details>

### TODO 2: Bring your own labels

Nothing forces us to use CIFAR-10's ten classes. Any list of sentences works as a set of options. The cell below asks CLIP a much coarser question about one image: animal or vehicle? (Every CIFAR-10 class is one or the other.)

Pick a different `my_image_index` (0 to 99), and write your own options. Try something fine-grained (`"a photo of a sports car."` vs. `"a photo of a taxi."`), or something CIFAR-10 doesn't cover at all.

In [ ]:
# TODO: pick an image (0 to 99) and write your own candidate sentences
my_image_index = 0
my_label_prompts = ["a photo of an animal.", "a photo of a vehicle."]

my_label_inputs = processor(text=my_label_prompts, padding=True, return_tensors="pt").to(device)
with torch.inference_mode():
    my_label_features = F.normalize(model.get_text_features(**my_label_inputs).pooler_output, dim=-1)
    my_probabilities = (logit_scale * image_features[my_image_index] @ my_label_features.T).softmax(dim=-1)

print("true CIFAR-10 label:", class_names[int(labels[my_image_index])])
for prompt, probability in sorted(zip(my_label_prompts, my_probabilities.tolist()), key=lambda pair: -pair[1]):
    print(f"{probability:.2f}  {prompt}")

The probabilities always add up to 1 across *your* list, even if none of your options fit the image. CLIP has to split its vote somewhere. Try giving it only bad options and watch it confidently pick one anyway.

### TODO 3: Which classes get mixed up?

Eight images in a gallery is a small window. The cell below counts every mistake in our 100-image sample by (true class, predicted class) and lists the most common ones.

In [ ]:
from collections import Counter

# TODO: change how many confusions to show
top_n = 5
mistakes = Counter(
    (class_names[int(truth)], class_names[int(guess)])
    for truth, guess in zip(labels, predictions.cpu())
    if truth != guess
)
for (truth, guess), count in mistakes.most_common(top_n):
    print(f"{count} x  true {truth} -> predicted {guess}")

Do the mix-ups make sense to you? Look for pairs that are genuinely hard at 32 × 32 pixels, and pairs that seem odd. Then go back to the template TODO and see if a different wording fixes any of them.

<details>
<summary>Hint</summary>

To look at the images behind a specific confusion, collect their positions and plot a few with `plt.imshow(images[i])`:

```python
[i for i in range(len(labels)) if class_names[int(labels[i])] == "cat" and class_names[int(predictions[i])] == "dog"]
```

</details>

---

**Up next, Notebook 06:** what if there's no fixed list of class names at all? We'll use the same shared space as a search engine, finding photos from a caption and captions from a photo.